In [ ]:
# Import libraries

import numpy as np
import pandas as pd
import scipy.stats as stats
from statsmodels.stats.proportion import proportions_ztest
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries loaded.")


Libraries loaded.


In [ ]:
from google.colab import files
import io

uploaded = files.upload()

df = pd.read_csv(io.BytesIO(uploaded["ObesityDataSet.csv"]))

print(f"Dataset shape: {df.shape}")
display(df.head())

Saving ObesityDataSet.csv to ObesityDataSet.csv
Dataset shape: (2111, 17)


,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad
0,Female,21.0,1.62,64.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,0.0,1.0,no,Public_Transportation,Normal_Weight
1,Female,21.0,1.52,56.0,yes,no,3.0,3.0,Sometimes,yes,3.0,yes,3.0,0.0,Sometimes,Public_Transportation,Normal_Weight
2,Male,23.0,1.80,77.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,2.0,1.0,Frequently,Public_Transportation,Normal_Weight
3,Male,27.0,1.80,87.0,no,no,3.0,3.0,Sometimes,no,2.0,no,2.0,0.0,Frequently,Walking,Overweight_Level_I
4,Male,22.0,1.78,89.8,no,no,2.0,1.0,Sometimes,no,2.0,no,0.0,0.0,Sometimes,Public_Transportation,Overweight_Level_II


In [ ]:
# Suskaičiuojame pradinius dublikatus
duplicate_rows_count = df.duplicated().sum()
print(f"Duplicates found: {duplicate_rows_count}")

if duplicate_rows_count > 0:
    # Pašaliname dublikatus ir išsaugome pakeitimus
    df = df.drop_duplicates(keep='first')
    print(f"Duplicates deleted succsesfully. New sample dimentions: {df.shape}")
else:
    print("No duplicates found.")

Duplicates found: 24
Duplicates deleted succsesfully. New sample dimentions: (2087, 17)


In [ ]:
ALPHA = 0.05

def decision(p_value, alpha=ALPHA):
    return "Reject H0" if p_value < alpha else "Do not reject H0"

def print_result(test_name, h0, h1, statistic_name, statistic, p_value, alpha=ALPHA):
    print("=" * 80)
    print(test_name)
    print("-" * 80)
    print("H0:", h0)
    print("H1:", h1)
    print(f"{statistic_name} = {statistic:.4f}")
    print(f"p-value = {p_value:.6f}")
    print(f"alpha = {alpha}")
    print("Decision:", decision(p_value, alpha))
    if p_value < alpha:
        print("Interpretation: There is statistically significant evidence against H0.")
    else:
        print("Interpretation: There is not sufficient statistical evidence against H0.")


In [ ]:

try:
    active_df = obesity_df
except NameError:
    try:
        active_df = df
    except NameError:
        import pandas as pd
        active_df = pd.read_csv("ObesityDataSet.csv")


water_female = active_df.loc[active_df['Gender'].str.lower().str.strip() == 'female', 'CH2O'].dropna()
water_male = active_df.loc[active_df['Gender'].str.lower().str.strip() == 'male', 'CH2O'].dropna()


mwu_stat, mwu_p = stats.mannwhitneyu(water_female, water_male, alternative='two-sided')

print_result(
    test_name = "Mann-Whitney U test (Water Intake Female vs Male)",
    h0 = "The distributions of water intake are equal",
    h1 = "The distributions of water intake are different",
    statistic_name = "U",
    statistic = mwu_stat,
    p_value = mwu_p
)

print(f"Female median = {water_female.median():.2f}")
print(f"Male median = {water_male.median():.2f}")

Mann-Whitney U test (Water Intake Female vs Male)
--------------------------------------------------------------------------------
H0: The distributions of water intake are equal
H1: The distributions of water intake are different
U = 477396.0000
p-value = 0.000001
alpha = 0.05
Decision: Reject H0
Interpretation: There is statistically significant evidence against H0.
Female median = 2.00
Male median = 2.00


In [ ]:
try:
    obesity_df
except NameError:
    import pandas as pd
    obesity_df = pd.read_csv("ObesityDataSet.csv")

group_yes = obesity_df.loc[obesity_df['family_history_with_overweight'].str.lower().str.strip() == 'yes', 'Weight'].dropna()
group_no = obesity_df.loc[obesity_df['family_history_with_overweight'].str.lower().str.strip() == 'no', 'Weight'].dropna()

print(f"With family history: n = {len(group_yes)}, mean = {group_yes.mean():.2f} kg, variance = {group_yes.var():.2f}")
print(f"Without family history: n = {len(group_no)}, mean = {group_no.mean():.2f} kg, variance = {group_no.var():.2f}")

stat_y, p_y = stats.shapiro(group_yes)
stat_n, p_n = stats.shapiro(group_no)

print_result(
    test_name = "Shapiro-Wilk test: Weight (With family history)",
    h0 = "The population distribution is normal",
    h1 = "The population distribution is not normal",
    statistic_name = "w",
    statistic = stat_y,
    p_value = p_y
)

print_result(
    test_name = "Shapiro-Wilk test: Weight (No family history)",
    h0 = "The population distribution is normal",
    h1 = "The population distribution is not normal",
    statistic_name = "w",
    statistic = stat_n,
    p_value = p_n
)


levene_stat, levene_p = stats.levene(group_yes, group_no, center='median')
print(f"\nLevene's test p-value: {levene_p:.6f}")

if levene_p < ALPHA:
    print("Reject H0: variances are statistically significantly different. Using Welch's t-test.")
    equal_vars = False
else:
    print("Do not reject H0: variances are not significantly different. Using classic t-test.")
    equal_vars = True


t_stat, p_val = stats.ttest_ind(group_yes, group_no, equal_var=equal_vars, alternative='two-sided')

test_title = "Welch's independent samples t-test" if not equal_vars else "Independent samples t-test (equal variances assumed)"

print_result(
    test_name = test_title,
    h0 = "μ_yes = μ_no",
    h1 = "μ_yes ≠ μ_no",
    statistic_name = "t",
    statistic = t_stat,
    p_value = p_val
)

With family history: n = 1726, mean = 92.73 kg, variance = 587.20
Without family history: n = 385, mean = 59.04 kg, variance = 201.12
Shapiro-Wilk test: Weight (With family history)
--------------------------------------------------------------------------------
H0: The population distribution is normal
H1: The population distribution is not normal
w = 0.9832
p-value = 0.000000
alpha = 0.05
Decision: Reject H0
Interpretation: There is statistically significant evidence against H0.
Shapiro-Wilk test: Weight (No family history)
--------------------------------------------------------------------------------
H0: The population distribution is normal
H1: The population distribution is not normal
w = 0.9330
p-value = 0.000000
alpha = 0.05
Decision: Reject H0
Interpretation: There is statistically significant evidence against H0.

Levene's test p-value: 0.000000
Reject H0: variances are statistically significantly different. Using Welch's t-test.
Welch's independent samples t-test
----------

In [ ]:

try:
    obesity_df
except NameError:
    import pandas as pd
    obesity_df = pd.read_csv("ObesityDataSet.csv")


contingency_table = pd.crosstab(obesity_df["Gender"], obesity_df["SMOKE"])


chi2_stat_s, p_val_s, dof_s, expected_s = stats.chi2_contingency(contingency_table)

print_result(
    test_name = "Chi-square test of independence (Gender vs SMOKE)",
    h0 = "Gender and SMOKE are independent",
    h1 = "Gender and SMOKE are dependent",
    statistic_name = "χ²",
    statistic = chi2_stat_s,
    p_value = p_val_s
)

print("\nObserved frequencies:")
display(contingency_table)

print("\nExpected frequencies:")
display(pd.DataFrame(expected_s, index=contingency_table.index, columns=contingency_table.columns))

Chi-square test of independence (Gender vs SMOKE)
--------------------------------------------------------------------------------
H0: Gender and SMOKE are independent
H1: Gender and SMOKE are dependent
χ² = 3.6150
p-value = 0.057260
alpha = 0.05
Decision: Do not reject H0
Interpretation: There is not sufficient statistical evidence against H0.

Observed frequencies:


SMOKE,no,yes
Gender,,
Female,1028,15
Male,1039,29



Expected frequencies:


SMOKE,no,yes
Gender,,
Female,1021.26054,21.73946
Male,1045.73946,22.26054
